# Ablation 3 — Model without the complete Transfer Entropy module

This experiment retains the Transformer and removes the complete branch designed to estimate Transfer Entropy.

Flow of the ablation:

`X → Transformer → channel projection → temporal pooling → classifier`

The following components are removed:

- nonlinear temporal filter `φ`;
- Takens reconstruction;
- Takens state projections;
- Rational Quadratic/Gaussian kernels;
- Transfer Entropy layer;
- diagonal removal and TE-matrix vectorization.

For direct comparison with the complete CTE-Net model:

- exactly the same participants and `folds.pkl` are used;
- the applicable Transformer, classifier, optimizer, scheduler, and early-stopping hyperparameters are retained;
- model weights are initialized with seed 42 in every fold, matching the effective protocol of the presented CTE-Net model;
- ten base seeds (`0`--`9`) are evaluated, and the training DataLoader uses `base_seed + fold_id`;
- every base seed runs the same five subject-wise folds, producing 50 trained models;
- metrics are first averaged across the five test folds within each seed and then summarized using the mean and sample standard deviation across the ten seeds;
- test predictions are retained at the window and participant levels for subsequent paired analyses;
- no majority voting is applied at the window level.

The deterministic temporal pooling uses 18 contiguous blocks of 28 samples (504 samples in total) and discards the final 8 samples from each window. This preserves 342 classifier input features without introducing trainable replacement parameters.


In [1]:
# ============================================================
# 1. IMPORTS, REPRODUCIBILIDAD Y DISPOSITIVO
# ============================================================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import pickle
import random
import shutil
import math
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)


def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    """Misma fijación de semillas utilizada en el cuaderno original."""
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


set_seed(42)

Dispositivo: cuda


In [2]:
class InspectableTransformerEncoder(nn.Module):
    """
    Réplica en idea del InspectableTransformerEncoder de TensorFlow/Keras.

    Flujo:
      input [B, T, C]
        -> proyección opcional a d_model
        -> MultiHeadAttention
        -> residual + LayerNorm
        -> FeedForward
        -> residual + LayerNorm

    Guarda los últimos attention scores para inspección.
    """
    def __init__(
        self,
        in_channels,
        d_model=None,
        num_heads=2,
        intermediate_dim=128,
        dropout=0.1,
        use_input_projection=True,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.d_model = d_model if d_model is not None else in_channels
        self.num_heads = num_heads
        self.intermediate_dim = intermediate_dim
        self.dropout_rate = dropout
        self.use_input_projection = use_input_projection

        if self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model={self.d_model} debe ser divisible entre num_heads={self.num_heads}"
            )

        # Proyección opcional para imitar tu adaptación en PyTorch
        if use_input_projection or self.d_model != in_channels:
            self.input_proj = nn.Linear(in_channels, self.d_model)
        else:
            self.input_proj = nn.Identity()

        # Self-attention inspeccionable
        self.self_attention = nn.MultiheadAttention(
            embed_dim=self.d_model,
            num_heads=self.num_heads,
            dropout=self.dropout_rate,
            batch_first=True,
        )

        # Normalización y dropout, igual idea que Keras
        self.self_attention_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)
        self.self_attention_dropout = nn.Dropout(self.dropout_rate)

        # Feedforward
        self.feedforward_intermediate_dense = nn.Linear(self.d_model, self.intermediate_dim)
        self.feedforward_output_dense = nn.Linear(self.intermediate_dim, self.d_model)
        self.feedforward_dropout = nn.Dropout(self.dropout_rate)
        self.feedforward_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)

        # Para guardar los scores
        self._last_attention_scores = None

    def forward(self, inputs, training=False, need_weights=True, attn_mask=None):
        """
        inputs: [B, T, C]
        returns: [B, T, d_model]
        """
        x = self.input_proj(inputs)  # [B, T, d_model]

        attention_output, attention_scores = self.self_attention(
            query=x,
            key=x,
            value=x,
            attn_mask=attn_mask,
            need_weights=need_weights,
            average_attn_weights=False,  # devuelve [B, H, T, T]
        )

        if need_weights:
            self._last_attention_scores = attention_scores
        else:
            self._last_attention_scores = None

        attention_output = self.self_attention_dropout(attention_output)
        attention_output = self.self_attention_layer_norm(x + attention_output)

        ff_output = self.feedforward_intermediate_dense(attention_output)
        ff_output = F.gelu(ff_output)
        ff_output = self.feedforward_output_dense(ff_output)
        ff_output = self.feedforward_dropout(ff_output)

        output = self.feedforward_layer_norm(attention_output + ff_output)
        return output

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado attention scores todavía. "
                "Haz un forward pass con need_weights=True."
            )
        return self._last_attention_scores

    def get_attention_weights(self):
        """
        Intenta imitar la idea de inspeccionar pesos Q, K, V y O.
        En PyTorch MultiheadAttention, Q/K/V suelen venir concatenados en in_proj_weight.
        """
        attn = self.self_attention

        weights_dict = {
            "in_proj_weight": attn.in_proj_weight.detach().cpu(),
            "in_proj_bias": attn.in_proj_bias.detach().cpu() if attn.in_proj_bias is not None else None,
            "out_proj_weight": attn.out_proj.weight.detach().cpu(),
            "out_proj_bias": attn.out_proj.bias.detach().cpu() if attn.out_proj.bias is not None else None,
        }
        return weights_dict

In [3]:
# ============================================================
# ABLACIÓN: MÓDULO DE TRANSFER ENTROPY ELIMINADO
# ============================================================
# En esta variante no se definen ni se instancian:
# - ChannelwiseTemporalFilter (φ)
# - TakensConv1D
# - KernelLayer
# - TransferEntropyLayer
# - RemoveDiagonalFlatten


In [4]:
# =========================================================
# MODELO DE ABLACIÓN: TRANSFORMER SIN MÓDULO TE
#
# Flujo:
# X -> Transformer -> proyección a canales
#   -> pooling temporal -> clasificador
# =========================================================

class DeterministicTemporalMeanPool(nn.Module):
    """
    Pooling temporal fijo y determinista.

    Convierte [B, C, T] en [B, C, output_size] usando bloques
    temporales contiguos y una media explícita con reshape.

    Para T=512 y output_size=18:
    - block_size = 512 // 18 = 28
    - used_samples = 18 * 28 = 504
    - se descartan únicamente las últimas 8 muestras
    """
    def __init__(self, input_size: int, output_size: int):
        super().__init__()

        self.input_size = int(input_size)
        self.output_size = int(output_size)

        if self.output_size <= 0:
            raise ValueError(
                "output_size debe ser mayor que cero."
            )

        if self.input_size < self.output_size:
            raise ValueError(
                "input_size debe ser mayor o igual que output_size."
            )

        self.block_size = (
            self.input_size // self.output_size
        )
        self.used_samples = (
            self.block_size * self.output_size
        )

        if self.block_size <= 0:
            raise ValueError(
                "El tamaño de bloque calculado no es válido."
            )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(
                f"Se esperaba [B, C, T], pero llegó {tuple(x.shape)}."
            )

        if x.shape[-1] != self.input_size:
            raise ValueError(
                f"Se esperaban {self.input_size} muestras temporales "
                f"y llegaron {x.shape[-1]}."
            )

        x_used = x[..., :self.used_samples]

        batch_size, channels, _ = x_used.shape

        x_blocks = x_used.reshape(
            batch_size,
            channels,
            self.output_size,
            self.block_size,
        )

        return x_blocks.mean(dim=-1)


class TransformerWithoutTEModule(nn.Module):
    def __init__(
        self,
        chans=19,
        samples=512,
        d_model=64,
        nhead=2,
        num_transformer_layers=1,
        dim_feedforward=128,
        transformer_dropout=0.1,
        clf_hidden=64,
        clf_dropout=0.3,
        use_pre_transformer_norm=True,
        use_post_transformer_norm=True,
        pooled_time_points=None,
    ):
        super().__init__()

        self.chans = int(chans)
        self.samples = int(samples)
        self.d_model = int(d_model)
        self.num_transformer_layers = int(
            num_transformer_layers
        )

        # Se eligen C-1 puntos temporales para mantener:
        # C * (C - 1) = 19 * 18 = 342 características,
        # el mismo tamaño que la matriz TE sin diagonal.
        self.pooled_time_points = int(
            pooled_time_points
            if pooled_time_points is not None
            else self.chans - 1
        )

        if self.pooled_time_points <= 0:
            raise ValueError(
                "pooled_time_points debe ser mayor que cero."
            )

        # 1) Normalización previa, idéntica al modelo completo.
        self.pre_transformer_norm = (
            nn.LayerNorm(self.chans)
            if use_pre_transformer_norm
            else nn.Identity()
        )

        # 2) Mismo stack Transformer del modelo completo.
        self.transformer_layers = nn.ModuleList(
            [
                InspectableTransformerEncoder(
                    in_channels=(
                        self.chans
                        if layer_id == 0
                        else self.d_model
                    ),
                    d_model=self.d_model,
                    num_heads=nhead,
                    intermediate_dim=dim_feedforward,
                    dropout=transformer_dropout,
                    use_input_projection=True,
                )
                for layer_id in range(
                    self.num_transformer_layers
                )
            ]
        )

        self.post_transformer_norm = (
            nn.LayerNorm(self.d_model)
            if use_post_transformer_norm
            else nn.Identity()
        )

        # 3) Misma proyección del espacio d_model a los canales EEG.
        # Se conserva como adaptador de salida del Transformer.
        self.proj_back = nn.Linear(
            self.d_model,
            self.chans,
        )

        # 4) Sustitución no entrenable y determinista
        # de toda la rama TE.
        #
        # Evita AdaptiveAvgPool1d porque su backward en CUDA puede
        # ser no determinista cuando torch.use_deterministic_algorithms(True).
        #
        # [B, C, 512] -> [B, C, 18]
        self.temporal_pool = DeterministicTemporalMeanPool(
            input_size=self.samples,
            output_size=self.pooled_time_points,
        )

        # 5) Clasificador con la misma dimensionalidad de entrada
        # que el modelo completo: 19 * 18 = 342.
        feature_dim = (
            self.chans
            * self.pooled_time_points
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                feature_dim,
                clf_hidden,
            ),
            nn.ReLU(),
            nn.Dropout(
                clf_dropout,
            ),
            nn.Linear(
                clf_hidden,
                1,
            ),
        )

    def forward(
        self,
        x,
        return_dict=True,
        need_attention=False,
    ):
        """
        x: tensor EEG [batch, canales, muestras].
        """
        if x.ndim != 3:
            raise ValueError(
                f"Se esperaba x con forma [B, C, T], "
                f"pero llegó {tuple(x.shape)}."
            )

        _, channels, samples = x.shape

        if channels != self.chans:
            raise ValueError(
                f"Se esperaban {self.chans} canales "
                f"y llegaron {channels}."
            )

        if samples != self.samples:
            raise ValueError(
                f"Se esperaban {self.samples} muestras "
                f"y llegaron {samples}."
            )

        # [B, C, T] -> [B, T, C]
        x_t = x.transpose(1, 2)

        x_t = self.pre_transformer_norm(
            x_t
        )

        # Stack Transformer.
        hidden = x_t
        attention_scores_all = (
            [] if need_attention else None
        )

        for layer in self.transformer_layers:
            hidden = layer(
                hidden,
                need_weights=need_attention,
            )

            if need_attention:
                attention_scores_all.append(
                    layer.get_attention_scores()
                )

        hidden = self.post_transformer_norm(
            hidden
        )

        # [B, T, d_model] -> [B, T, C] -> [B, C, T]
        transformer_channels = self.proj_back(
            hidden
        ).transpose(1, 2)

        # -------------------------------------------------
        # ABLACIÓN:
        # aquí termina el Transformer y se omite por completo:
        # φ -> Takens -> kernels -> TE -> quitar diagonal.
        # -------------------------------------------------
        pooled = self.temporal_pool(
            transformer_channels
        )

        features = pooled.flatten(
            start_dim=1
        )

        logits = self.classifier(
            features
        ).squeeze(-1)

        probabilities = torch.sigmoid(
            logits
        )

        if return_dict:
            return {
                "logits": logits,
                "probs": probabilities,
                "features": features,
                "transformer_features": (
                    transformer_channels
                ),
                "pooled_features": pooled,
                "attention_scores": (
                    attention_scores_all
                    if need_attention
                    else None
                ),
                # Se mantienen estas claves en None para dejar
                # explícito que el módulo TE no existe.
                "Xf": transformer_channels,
                "phi": None,
                "T": None,
            }

        return logits


# =========================================================
# LOSS
# =========================================================
def classification_loss(logits, y_true):
    y_true = y_true.float().view(-1)

    return F.binary_cross_entropy_with_logits(
        logits,
        y_true,
    )


In [5]:
# ============================================================
# 2. RUTAS, FOLDS Y CARGA DE LA BASE DE DATOS
# ============================================================
# Kaggle path used by the principal CTE-Net notebooks. A local fallback is
# retained so the notebook can also run outside Kaggle.
KAGGLE_DATA_ROOT = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH"
)
LOCAL_DATA_ROOT = Path("..") / "data" / "raw"

configured_data_root = os.environ.get("CTE_NET_DATA_ROOT")
candidate_data_roots = []
if configured_data_root:
    candidate_data_roots.append(Path(configured_data_root))
candidate_data_roots.extend([KAGGLE_DATA_ROOT, LOCAL_DATA_ROOT])

DATA_ROOT = next(
    (
        candidate
        for candidate in candidate_data_roots
        if (candidate / "folds.pkl").is_file()
        and (candidate / "ieee" / "ADHD_group").is_dir()
        and (candidate / "ieee" / "Control_group").is_dir()
    ),
    None,
)

if DATA_ROOT is None:
    checked_paths = "\n".join(
        f" - {candidate}" for candidate in candidate_data_roots
    )
    raise FileNotFoundError(
        "The EEG dataset root could not be located. The following "
        "paths were checked:\n"
        f"{checked_paths}\n"
        "In Kaggle, add the dataset 'daprosero/mi-tdah-dataset'. "
        "Alternatively, set CTE_NET_DATA_ROOT to the directory that "
        "contains folds.pkl and ieee/."
    )

ADHD_DIR = DATA_ROOT / "ieee" / "ADHD_group"
CONTROL_DIR = DATA_ROOT / "ieee" / "Control_group"
FOLDS_PATH = DATA_ROOT / "folds.pkl"

print("Dataset root:", DATA_ROOT)
print("Folds file:", FOLDS_PATH)

SEGMENT_SAMPLES = 512
OVERLAP = 0.50
EXPECTED_CHANS = 19


def canonical_subject_id(value) -> str:
    """Normaliza los IDs para comparar nombres de archivos y folds.pkl."""
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    return Path(str(value)).stem.strip()


# ------------------------------------------------------------
# Cargar primero las particiones.
# folds.pkl es la única fuente que decide qué sujetos participan.
# No se excluye manualmente v36p, v56p ni ningún otro sujeto.
# ------------------------------------------------------------
with open(FOLDS_PATH, "rb") as file:
    folds_raw = pickle.load(file)

folds = []
for fold_id, fold in enumerate(folds_raw):
    if len(fold) != 3:
        raise ValueError(
            f"El fold {fold_id} debe contener train, validation y test."
        )

    train_subjects, val_subjects, test_subjects = fold

    train_subjects = [canonical_subject_id(x) for x in train_subjects]
    val_subjects = [canonical_subject_id(x) for x in val_subjects]
    test_subjects = [canonical_subject_id(x) for x in test_subjects]

    folds.append((train_subjects, val_subjects, test_subjects))

# Unión de todos los sujetos incluidos en las particiones.
SUBJECTS_IN_FOLDS = {
    canonical_subject_id(subject)
    for train_subjects, val_subjects, test_subjects in folds
    for subject in (train_subjects + val_subjects + test_subjects)
}

print("Sujetos únicos definidos por folds.pkl:", len(SUBJECTS_IN_FOLDS))
print("¿v36p está en folds.pkl?:", "v36p" in SUBJECTS_IN_FOLDS)


def load_mat_eeg(path: Path, expected_chans: int = 19) -> np.ndarray:
    """
    Carga una matriz EEG y la devuelve con forma [canales, tiempo].
    Busca una variable numérica bidimensional que contenga 19 canales.
    """
    content = scipy.io.loadmat(path)

    candidates = []
    for key, value in content.items():
        if key.startswith("__"):
            continue

        array = np.asarray(value)
        if array.ndim == 2 and np.issubdtype(array.dtype, np.number):
            candidates.append((key, array))

    if not candidates:
        raise ValueError(f"No se encontró una matriz EEG 2D en {path}")

    candidates.sort(
        key=lambda item: (
            expected_chans not in item[1].shape,
            -item[1].size,
        )
    )

    for key, array in candidates:
        if array.shape[0] == expected_chans:
            eeg = array
            break
        if array.shape[1] == expected_chans:
            eeg = array.T
            break
    else:
        shapes = {key: arr.shape for key, arr in candidates}
        raise ValueError(
            f"Ninguna matriz de {path.name} contiene {expected_chans} canales. "
            f"Variables encontradas: {shapes}"
        )

    if not np.isfinite(eeg).all():
        raise ValueError(f"La señal contiene NaN o Inf: {path}")

    return np.ascontiguousarray(eeg, dtype=np.float32)


def load_group(
    folder: Path,
    label: int,
    allowed_subjects: set[str],
) -> tuple[dict, dict]:
    """
    Carga solamente los sujetos incluidos en folds.pkl.

    No existe una lista manual de exclusión: cualquier sujeto que no forme
    parte de las particiones simplemente no se carga.
    """
    allowed = {
        canonical_subject_id(subject)
        for subject in allowed_subjects
    }

    signals = {}
    labels = {}

    if not folder.exists():
        raise FileNotFoundError(f"No existe la carpeta: {folder}")

    mat_files = sorted(folder.glob("*.mat"))
    if not mat_files:
        raise FileNotFoundError(f"No se encontraron archivos .mat en: {folder}")

    for mat_path in mat_files:
        subject = canonical_subject_id(mat_path.name)

        # folds.pkl decide si el sujeto participa.
        if subject not in allowed:
            continue

        signals[subject] = load_mat_eeg(
            mat_path,
            expected_chans=EXPECTED_CHANS,
        )
        labels[subject] = int(label)

    return signals, labels


def segment_signals(
    signals: dict,
    labels_by_subject: dict,
    segment_samples: int = 512,
    overlap: float = 0.50,
):
    """Segmenta cada sujeto sin mezclar sujetos ni etiquetas."""
    if not 0 <= overlap < 1:
        raise ValueError("overlap debe estar en el intervalo [0, 1).")

    step = int(round(segment_samples * (1.0 - overlap)))
    if step <= 0:
        raise ValueError("El paso de segmentación debe ser mayor que cero.")

    X = []
    y = []
    subjects = []
    window_ids = []

    for subject in sorted(signals):
        eeg = signals[subject]
        _, n_samples = eeg.shape

        if n_samples < segment_samples:
            print(
                f"Advertencia: {subject} tiene {n_samples} muestras y "
                f"no genera ventanas de {segment_samples}."
            )
            continue

        window_number = 0
        for start in range(0, n_samples - segment_samples + 1, step):
            stop = start + segment_samples

            X.append(eeg[:, start:stop])
            y.append(labels_by_subject[subject])
            subjects.append(subject)
            window_ids.append(window_number)

            window_number += 1

    if not X:
        raise RuntimeError("La segmentación no produjo ninguna ventana.")

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(subjects, dtype=object),
        np.asarray(window_ids, dtype=np.int64),
    )


def load_segmented_dataset():
    adhd_signals, adhd_labels = load_group(
        ADHD_DIR,
        label=1,
        allowed_subjects=SUBJECTS_IN_FOLDS,
    )
    control_signals, control_labels = load_group(
        CONTROL_DIR,
        label=0,
        allowed_subjects=SUBJECTS_IN_FOLDS,
    )

    repeated = set(adhd_signals).intersection(control_signals)
    if repeated:
        raise ValueError(
            f"Hay IDs repetidos entre ADHD y control: {sorted(repeated)}"
        )

    signals = {**control_signals, **adhd_signals}
    labels = {**control_labels, **adhd_labels}

    missing_files = SUBJECTS_IN_FOLDS - set(signals)
    if missing_files:
        raise FileNotFoundError(
            "Los siguientes sujetos aparecen en folds.pkl, pero no se "
            f"encontraron en ADHD_group o Control_group: {sorted(missing_files)}"
        )

    X, y, subjects, window_ids = segment_signals(
        signals=signals,
        labels_by_subject=labels,
        segment_samples=SEGMENT_SAMPLES,
        overlap=OVERLAP,
    )

    return X, y, subjects, window_ids


X, y, sbjs, window_ids = load_segmented_dataset()

print("\nForma de X:", X.shape)
print("Forma de y:", y.shape)
print("Canales:", X.shape[1])
print("Muestras por ventana:", X.shape[2])
print("Sujetos cargados:", len(np.unique(sbjs)))
print("Control:", len(np.unique(sbjs[y == 0])))
print("ADHD:", len(np.unique(sbjs[y == 1])))
print("Número de folds:", len(folds))


def validate_folds(folds, available_subjects):
    available = set(map(canonical_subject_id, available_subjects))

    for fold_id, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        train_set = set(train_subjects)
        val_set = set(val_subjects)
        test_set = set(test_subjects)

        if train_set & val_set or train_set & test_set or val_set & test_set:
            raise ValueError(
                f"Fold {fold_id}: existe solapamiento entre train/val/test."
            )

        missing = (train_set | val_set | test_set) - available
        if missing:
            raise ValueError(
                f"Fold {fold_id}: sujetos de folds.pkl no encontrados "
                f"en las carpetas: {sorted(missing)}"
            )

        print(
            f"Fold {fold_id}: "
            f"train={len(train_set)}, val={len(val_set)}, test={len(test_set)}"
        )


validate_folds(folds, np.unique(sbjs))


Dataset root: /kaggle/input/datasets/daprosero/mi-tdah-dataset/MI_TDAH_Dataset/TDAH
Folds file: /kaggle/input/datasets/daprosero/mi-tdah-dataset/MI_TDAH_Dataset/TDAH/folds.pkl
Sujetos únicos definidos por folds.pkl: 120
¿v36p está en folds.pkl?: False

Forma de X: (8213, 19, 512)
Forma de y: (8213,)
Canales: 19
Muestras por ventana: 512
Sujetos cargados: 120
Control: 60
ADHD: 60
Número de folds: 5
Fold 0: train=76, val=20, test=24
Fold 1: train=76, val=20, test=24
Fold 2: train=76, val=20, test=24
Fold 3: train=76, val=20, test=24
Fold 4: train=76, val=20, test=24


In [6]:
# ============================================================
# 6. ENTRENAMIENTO Y EVALUACIÓN EXCLUSIVAMENTE POR VENTANA
# ============================================================
def ensure_binary_labels(labels) -> np.ndarray:
    labels = np.asarray(labels).reshape(-1)

    unique = np.unique(labels)
    if not set(unique.tolist()).issubset({0, 1, 0.0, 1.0}):
        raise ValueError(
            f"Se esperaban etiquetas binarias 0/1; llegaron {unique}"
        )

    return labels.astype(np.int64)


def build_model(model_args: dict, seed: int = 42) -> nn.Module:
    """
    Crea un modelo nuevo de la ablación sin cargar pesos.

    Se reproduce el comportamiento efectivo del cuaderno original:
    los pesos se inicializan siempre con seed=42.
    """
    set_seed(42)

    return TransformerWithoutTEModule(
        chans=model_args["Chans"],
        samples=model_args["Samples"],
        d_model=model_args["d_model"],
        nhead=model_args["nhead"],
        num_transformer_layers=model_args[
            "num_transformer_layers"
        ],
        dim_feedforward=model_args[
            "dim_feedforward"
        ],
        transformer_dropout=model_args[
            "transformer_dropout"
        ],
        clf_hidden=model_args["clf_hidden"],
        clf_dropout=model_args["clf_dropout"],
        use_pre_transformer_norm=model_args.get(
            "use_pre_transformer_norm",
            True,
        ),
        use_post_transformer_norm=model_args.get(
            "use_post_transformer_norm",
            True,
        ),
        pooled_time_points=model_args.get(
            "pooled_time_points",
            model_args["Chans"] - 1,
        ),
    )

def build_optimizer_scheduler(model, compile_cfg):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=compile_cfg["learning_rate"],
        weight_decay=compile_cfg["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=compile_cfg["schedule_factor"],
        patience=compile_cfg["schedule_patience"],
        min_lr=compile_cfg["min_lr"],
    )

    return optimizer, scheduler


def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for xb, yb in loader:
        xb = xb.to(
            device=device,
            dtype=torch.float32,
        )
        yb = yb.to(
            device=device,
            dtype=torch.float32,
        ).view(-1)

        optimizer.zero_grad()

        output = model(
            xb,
            return_dict=True,
            need_attention=False,
        )
        loss = classification_loss(
            output["logits"],
            yb,
        )

        loss.backward()
        optimizer.step()

        batch_size_current = xb.shape[0]
        total_loss += float(loss.item()) * batch_size_current
        total_examples += batch_size_current

    return total_loss / max(total_examples, 1)


@torch.no_grad()
def predict_windows(model, loader, device):
    """Obtiene pérdida y predicciones para cada ventana EEG."""
    model.eval()

    total_loss = 0.0
    total_examples = 0
    y_true_all = []
    y_prob_all = []

    for xb, yb in loader:
        xb = xb.to(
            device=device,
            dtype=torch.float32,
        )
        yb = yb.to(
            device=device,
            dtype=torch.float32,
        ).view(-1)

        output = model(
            xb,
            return_dict=True,
            need_attention=False,
        )

        logits = output["logits"]
        probabilities = torch.sigmoid(logits)
        loss = classification_loss(logits, yb)

        batch_size_current = xb.shape[0]
        total_loss += float(loss.item()) * batch_size_current
        total_examples += batch_size_current

        y_true_all.append(
            yb.detach().cpu().numpy()
        )
        y_prob_all.append(
            probabilities.detach().cpu().numpy()
        )

    y_true = np.concatenate(
        y_true_all
    ).astype(np.int64)

    y_prob = np.concatenate(
        y_prob_all
    ).astype(np.float64)

    y_pred = (
        y_prob >= 0.5
    ).astype(np.int64)

    return {
        "loss": float(
            total_loss / max(total_examples, 1)
        ),
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }


def binary_window_metrics(y_true, y_pred, y_prob) -> dict:
    """Calcula todas las métricas utilizando una observación por ventana."""
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )
    y_pred = np.asarray(
        y_pred,
        dtype=np.int64,
    )
    y_prob = np.asarray(
        y_prob,
        dtype=np.float64,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    try:
        auc_value = roc_auc_score(
            y_true,
            y_prob,
        )
    except ValueError:
        auc_value = np.nan

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "kappa": float(
            cohen_kappa_score(
                y_true,
                y_pred,
            )
        ),
        "auc": float(auc_value),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "n_windows": int(len(y_true)),
    }


def make_window_predictions(
    subjects,
    window_numbers,
    predictions,
    split,
    fold_id,
) -> pd.DataFrame:
    """Construye el archivo de predicciones sin agrupar por sujeto."""
    n_predictions = len(
        predictions["y_true"]
    )

    if len(subjects) != n_predictions:
        raise ValueError(
            "Los identificadores de sujeto no coinciden "
            "con el número de predicciones."
        )

    if len(window_numbers) != n_predictions:
        raise ValueError(
            "Los identificadores de ventana no coinciden "
            "con el número de predicciones."
        )

    return pd.DataFrame(
        {
            "fold": int(fold_id),
            "split": split,
            "subject": np.asarray(
                subjects,
                dtype=object,
            ),
            "window_id": np.asarray(
                window_numbers
            ),
            "y_true": predictions["y_true"],
            "y_prob": predictions["y_prob"],
            "y_pred": predictions["y_pred"],
            "correct": (
                predictions["y_true"]
                == predictions["y_pred"]
            ).astype(np.int64),
        }
    )


def mean_std_window_metrics_over_folds(
    fold_metrics_df: pd.DataFrame,
    prefix: str = "test_window",
) -> dict:
    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "f1",
        "sensitivity",
        "specificity",
        "precision",
        "kappa",
        "auc",
    ]

    output = {}

    for metric in metric_names:
        column = f"{prefix}_{metric}"
        values = fold_metrics_df[
            column
        ].to_numpy(dtype=float)

        output[metric] = {
            "mean": float(
                np.nanmean(values)
            ),
            "std_population": float(
                np.nanstd(
                    values,
                    ddof=0,
                )
            ),
            "std_sample": (
                float(
                    np.nanstd(
                        values,
                        ddof=1,
                    )
                )
                if len(values) > 1
                else 0.0
            ),
        }

    return output


def train_fixed_hps_5fold(
    X,
    y,
    subjects,
    window_ids,
    folds,
    model_args,
    compile_cfg,
    output_dir,
    epochs=100,
    batch_size=16,
    seed=42,
    early_stopping_patience=25,
    force_retrain=True,
):
    """
    Entrena un modelo nuevo en cada fold con hiperparámetros fijos.

    Protocolo:
    - no ejecuta Optuna;
    - no reutiliza pesos anteriores;
    - selecciona la mejor época mediante validation loss;
    - restaura el mejor checkpoint del mismo fold;
    - evalúa validation y test exclusivamente por ventana;
    - no aplica voto mayoritario.
    """
    output_dir = Path(output_dir)

    if force_retrain and output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    X = np.asarray(
        X,
        dtype=np.float32,
    )
    y = ensure_binary_labels(y)

    subjects = np.asarray(
        [
            canonical_subject_id(value)
            for value in subjects
        ],
        dtype=object,
    )

    window_ids = np.asarray(window_ids)

    if not (
        len(X)
        == len(y)
        == len(subjects)
        == len(window_ids)
    ):
        raise ValueError(
            "X, y, subjects y window_ids deben tener "
            "el mismo número de ventanas."
        )

    config = {
        "evaluation_level": "window",
        "majority_vote": False,
        "base_seed": int(seed),
        "model_initialization_seed": 42,
        "dataloader_seed_rule": "base_seed + fold_id",
        "std_rule": "population_ddof_0",
        "epochs": int(epochs),
        "batch_size": int(batch_size),
        "early_stopping_patience": int(
            early_stopping_patience
        ),
        "model": "TransformerWithoutTEModule",
        "ablation": "complete_te_module_removed",
        "removed_components": [
            "ChannelwiseTemporalFilter_phi",
            "TakensConv1D",
            "Takens_state_projections",
            "KernelLayer_x",
            "KernelLayer_y_past",
            "KernelLayer_y_present",
            "TransferEntropyLayer",
            "RemoveDiagonalFlatten"
        ],
        "replacement": {
            "type": "DeterministicTemporalMeanPool",
            "pooling_rule": "18 contiguous blocks of 28 samples; last 8 samples discarded",
            "output_time_points": int(
                model_args.get(
                    "pooled_time_points",
                    model_args["Chans"] - 1
                )
            ),
            "trainable": False,
            "feature_dimension": int(
                model_args["Chans"]
                * model_args.get(
                    "pooled_time_points",
                    model_args["Chans"] - 1
                )
            )
        },
        "model_args": model_args,
        "compile_cfg": compile_cfg,
    }

    with open(
        output_dir / "fixed_hps_config.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            config,
            file,
            indent=2,
        )

    fold_rows = []
    all_test_window_predictions = []

    for fold_id, (
        train_subjects,
        val_subjects,
        test_subjects,
    ) in enumerate(folds):
        print("\n" + "=" * 88)
        print(
            f"FOLD {fold_id + 1}/{len(folds)}"
        )
        print("=" * 88)

        fold_dir = (
            output_dir
            / f"fold_{fold_id}"
        )
        fold_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        metrics_path = (
            fold_dir
            / "fold_metrics.json"
        )
        weights_path = (
            fold_dir
            / "best_state_dict.pt"
        )
        test_predictions_path = (
            fold_dir
            / "test_window_predictions.csv"
        )

        # Reutiliza un fold únicamente cuando fue terminado antes
        # y force_retrain=False. No carga pesos de Optuna.
        if (
            not force_retrain
            and metrics_path.exists()
            and test_predictions_path.exists()
        ):
            print(
                f"Fold {fold_id} ya terminado; "
                "se cargan sus resultados guardados."
            )

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as file:
                fold_row = json.load(file)

            test_window_df = pd.read_csv(
                test_predictions_path
            )

            fold_rows.append(fold_row)
            all_test_window_predictions.append(
                test_window_df
            )
            continue

        train_set = {
            canonical_subject_id(value)
            for value in train_subjects
        }
        val_set = {
            canonical_subject_id(value)
            for value in val_subjects
        }
        test_set = {
            canonical_subject_id(value)
            for value in test_subjects
        }

        # Verificar que las particiones no se solapen.
        if train_set & val_set:
            raise ValueError(
                f"Fold {fold_id}: train y validation "
                "comparten sujetos."
            )

        if train_set & test_set:
            raise ValueError(
                f"Fold {fold_id}: train y test "
                "comparten sujetos."
            )

        if val_set & test_set:
            raise ValueError(
                f"Fold {fold_id}: validation y test "
                "comparten sujetos."
            )

        train_idx = np.flatnonzero(
            np.isin(
                subjects,
                list(train_set),
            )
        )
        val_idx = np.flatnonzero(
            np.isin(
                subjects,
                list(val_set),
            )
        )
        test_idx = np.flatnonzero(
            np.isin(
                subjects,
                list(test_set),
            )
        )

        if min(
            len(train_idx),
            len(val_idx),
            len(test_idx),
        ) == 0:
            raise RuntimeError(
                f"Fold {fold_id}: algún split no contiene ventanas. "
                f"train={len(train_idx)}, "
                f"validation={len(val_idx)}, "
                f"test={len(test_idx)}"
            )

        print(
            "Ventanas | "
            f"train={len(train_idx)}, "
            f"validation={len(val_idx)}, "
            f"test={len(test_idx)}"
        )

        print(
            "Sujetos  | "
            f"train={len(train_set)}, "
            f"validation={len(val_set)}, "
            f"test={len(test_set)}"
        )

        fold_seed = int(
            seed + fold_id
        )
        set_seed(fold_seed)

        train_dataset = TensorDataset(
            torch.from_numpy(
                X[train_idx]
            ).float(),
            torch.from_numpy(
                y[train_idx]
            ).float(),
        )

        val_dataset = TensorDataset(
            torch.from_numpy(
                X[val_idx]
            ).float(),
            torch.from_numpy(
                y[val_idx]
            ).float(),
        )

        test_dataset = TensorDataset(
            torch.from_numpy(
                X[test_idx]
            ).float(),
            torch.from_numpy(
                y[test_idx]
            ).float(),
        )

        train_generator = torch.Generator()
        train_generator.manual_seed(
            fold_seed
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
            num_workers=0,
            generator=train_generator,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        model = build_model(
            model_args,
            seed=fold_seed,
        ).to(DEVICE)

        optimizer, scheduler = (
            build_optimizer_scheduler(
                model,
                compile_cfg,
            )
        )

        best_val_loss = np.inf
        best_epoch = 0
        patience_counter = 0
        history_rows = []

        for epoch in range(
            1,
            epochs + 1,
        ):
            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                DEVICE,
            )

            val_predictions = predict_windows(
                model,
                val_loader,
                DEVICE,
            )

            val_loss = val_predictions["loss"]

            scheduler.step(val_loss)

            current_lr = float(
                optimizer.param_groups[0]["lr"]
            )

            val_metrics = binary_window_metrics(
                val_predictions["y_true"],
                val_predictions["y_pred"],
                val_predictions["y_prob"],
            )

            history_rows.append(
                {
                    "epoch": int(epoch),
                    "train_loss": float(
                        train_loss
                    ),
                    "val_loss": float(
                        val_loss
                    ),
                    "val_accuracy_window": (
                        val_metrics["accuracy"]
                    ),
                    "val_f1_window": (
                        val_metrics["f1"]
                    ),
                    "val_auc_window": (
                        val_metrics["auc"]
                    ),
                    "learning_rate": current_lr,
                }
            )

            improved = (
                val_loss
                < best_val_loss - 1e-4
            )

            if improved:
                best_val_loss = float(
                    val_loss
                )
                best_epoch = int(epoch)
                patience_counter = 0

                torch.save(
                    model.state_dict(),
                    weights_path,
                )
            else:
                patience_counter += 1

            if (
                epoch == 1
                or epoch % 5 == 0
                or improved
            ):
                print(
                    f"Epoch {epoch:03d} | "
                    f"train_loss={train_loss:.5f} | "
                    f"val_loss={val_loss:.5f} | "
                    f"val_acc_window="
                    f"{val_metrics['accuracy']:.4f} | "
                    f"patience={patience_counter}/"
                    f"{early_stopping_patience}"
                )

            if (
                patience_counter
                >= early_stopping_patience
            ):
                print(
                    f"Early stopping en epoch {epoch}."
                )
                break

        pd.DataFrame(
            history_rows
        ).to_csv(
            fold_dir / "history.csv",
            index=False,
        )

        if not weights_path.exists():
            raise RuntimeError(
                f"No se guardaron pesos para el fold {fold_id}."
            )

        # Restaurar el mejor estado generado en este mismo fold.
        best_state = torch.load(
            weights_path,
            map_location=DEVICE,
        )
        model.load_state_dict(
            best_state
        )

        val_predictions = predict_windows(
            model,
            val_loader,
            DEVICE,
        )
        test_predictions = predict_windows(
            model,
            test_loader,
            DEVICE,
        )

        val_window_df = make_window_predictions(
            subjects=subjects[val_idx],
            window_numbers=window_ids[val_idx],
            predictions=val_predictions,
            split="validation",
            fold_id=fold_id,
        )

        test_window_df = make_window_predictions(
            subjects=subjects[test_idx],
            window_numbers=window_ids[test_idx],
            predictions=test_predictions,
            split="test",
            fold_id=fold_id,
        )

        val_metrics = binary_window_metrics(
            val_window_df["y_true"],
            val_window_df["y_pred"],
            val_window_df["y_prob"],
        )

        test_metrics = binary_window_metrics(
            test_window_df["y_true"],
            test_window_df["y_pred"],
            test_window_df["y_prob"],
        )

        val_window_df.to_csv(
            fold_dir
            / "validation_window_predictions.csv",
            index=False,
        )

        test_window_df.to_csv(
            test_predictions_path,
            index=False,
        )

        fold_row = {
            "fold": int(fold_id),
            "model_initialization_seed": 42,
            "dataloader_seed": int(fold_seed),
            "best_epoch": int(best_epoch),
            "best_val_loss": float(
                best_val_loss
            ),
        }

        for metric, value in val_metrics.items():
            fold_row[
                f"val_window_{metric}"
            ] = value

        for metric, value in test_metrics.items():
            fold_row[
                f"test_window_{metric}"
            ] = value

        with open(
            metrics_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                fold_row,
                file,
                indent=2,
            )

        fold_rows.append(fold_row)
        all_test_window_predictions.append(
            test_window_df
        )

        print(
            f"Fold {fold_id} terminado | "
            f"Test por ventana: "
            f"Acc={test_metrics['accuracy']:.4f}, "
            f"F1={test_metrics['f1']:.4f}, "
            f"AUC={test_metrics['auc']:.4f}"
        )

        del model
        del optimizer
        del scheduler

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_metrics_df = (
        pd.DataFrame(fold_rows)
        .sort_values("fold")
        .reset_index(drop=True)
    )

    test_windows_df = pd.concat(
        all_test_window_predictions,
        ignore_index=True,
    )

    fold_metrics_df.to_csv(
        output_dir
        / "metrics_by_fold_window_level.csv",
        index=False,
    )

    test_windows_df.to_csv(
        output_dir
        / "test_window_predictions_all_folds.csv",
        index=False,
    )

    fold_summary = (
        mean_std_window_metrics_over_folds(
            fold_metrics_df,
            prefix="test_window",
        )
    )

    pooled_metrics = binary_window_metrics(
        test_windows_df["y_true"],
        test_windows_df["y_pred"],
        test_windows_df["y_prob"],
    )

    summary = {
        "model": "TransformerWithoutTEModule",
        "evaluation_level": "window",
        "majority_vote": False,
        "ablation": "complete_te_module_removed",
        "evaluation_protocol": (
            "Transformer retained; complete TE branch removed; "
            "non-trainable deterministic temporal mean pooling used to preserve "
            "342 classifier input features; fixed hyperparameters; "
            "five subject-wise folds; model weights initialized with "
            "seed 42 in every fold; DataLoader shuffled with "
            "base_seed + fold_id; best epoch selected using validation "
            "loss; test metrics calculated directly over EEG windows; "
            "population standard deviation across folds"
        ),
        "n_folds": int(len(folds)),
        "window_metrics_mean_std_over_folds": (
            fold_summary
        ),
        "pooled_window_metrics": (
            pooled_metrics
        ),
    }

    with open(
        output_dir / "summary_5fold_window_level.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    return {
        "fold_metrics": fold_metrics_df,
        "test_window_predictions": (
            test_windows_df
        ),
        "summary": summary,
        "output_dir": str(output_dir),
    }


In [7]:
# ============================================================
# 7. FIXED HYPERPARAMETERS AND TEN-SEED EXECUTION
#    TRANSFORMER WITHOUT THE COMPLETE TE MODULE
# ============================================================
# Only hyperparameters that remain applicable are retained.
# No new hyperparameter search is performed.

if X.ndim != 3 or X.shape[1] != 19 or X.shape[2] != 512:
    raise ValueError(
        "For direct comparison with the complete model, "
        "X was expected to have shape [N, 19, 512], "
        f"but received {X.shape}."
    )

BEST_COMPATIBLE_PARAMS = {
    "d_model": 32,
    "nhead": 2,
    "num_transformer_layers": 2,
    "dim_feedforward": 256,
    "transformer_dropout": 0.4,
    "clf_hidden": 128,
    "clf_dropout": 0.1,
    "learning_rate": 0.001,
    "weight_decay": 1.0394905157104865e-05,
    "schedule_factor": 0.1,
    "schedule_patience": 11,
    "min_lr": 2.4660476577131286e-06,
}

MODEL_ARGS = {
    "Chans": 19,
    "Samples": 512,
    "d_model": BEST_COMPATIBLE_PARAMS["d_model"],
    "nhead": BEST_COMPATIBLE_PARAMS["nhead"],
    "num_transformer_layers": BEST_COMPATIBLE_PARAMS[
        "num_transformer_layers"
    ],
    "dim_feedforward": BEST_COMPATIBLE_PARAMS[
        "dim_feedforward"
    ],
    "transformer_dropout": BEST_COMPATIBLE_PARAMS[
        "transformer_dropout"
    ],
    "clf_hidden": BEST_COMPATIBLE_PARAMS["clf_hidden"],
    "clf_dropout": BEST_COMPATIBLE_PARAMS["clf_dropout"],
    "use_pre_transformer_norm": True,
    "use_post_transformer_norm": True,
    # 19 channels x 18 temporal points = 342 features.
    "pooled_time_points": 18,
}

COMPILE_CFG = {
    "learning_rate": BEST_COMPATIBLE_PARAMS["learning_rate"],
    "weight_decay": BEST_COMPATIBLE_PARAMS["weight_decay"],
    "schedule_factor": BEST_COMPATIBLE_PARAMS["schedule_factor"],
    "schedule_patience": BEST_COMPATIBLE_PARAMS["schedule_patience"],
    "min_lr": BEST_COMPATIBLE_PARAMS["min_lr"],
}

BASE_SEEDS = list(range(10))

if os.environ.get("CTE_NET_SMOKE_TEST") == "1":
    BASE_SEEDS = [0]

if Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path(
        "/kaggle/working/ablation_without_te_repeated_10seeds"
    )
else:
    OUTPUT_ROOT = (
        Path("..")
        / "results"
        / "ablation_without_te_repeated_10seeds"
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# False permits safe resumption: completed folds are reused and missing
# folds are trained. Set True only when every stored fold must be replaced.
FORCE_RETRAIN = False

METRIC_COLUMNS = [
    "test_window_accuracy",
    "test_window_balanced_accuracy",
    "test_window_sensitivity",
    "test_window_specificity",
    "test_window_precision",
    "test_window_f1",
    "test_window_auc",
    "test_window_kappa",
]

all_fold_frames = []
all_window_prediction_frames = []
run_summaries = []

for repeat_id, base_seed in enumerate(BASE_SEEDS):
    print("\n" + "=" * 96)
    print(
        f"REPETITION {repeat_id + 1}/{len(BASE_SEEDS)} | "
        f"base seed = {base_seed}"
    )
    print("=" * 96)

    seed_output_dir = OUTPUT_ROOT / f"seed_{base_seed}"

    seed_results = train_fixed_hps_5fold(
        X=X,
        y=y,
        subjects=sbjs,
        window_ids=window_ids,
        folds=folds,
        model_args=MODEL_ARGS,
        compile_cfg=COMPILE_CFG,
        output_dir=seed_output_dir,
        epochs=100,
        batch_size=16,
        seed=base_seed,
        early_stopping_patience=25,
        force_retrain=FORCE_RETRAIN,
    )

    fold_frame = seed_results["fold_metrics"].copy()
    fold_frame.insert(0, "model", "Without TE Module")
    fold_frame.insert(1, "ablation", "complete_te_module_removed")
    fold_frame.insert(2, "seed", int(base_seed))
    fold_frame.insert(3, "repeat_id", int(repeat_id))
    all_fold_frames.append(fold_frame)

    window_frame = seed_results["test_window_predictions"].copy()
    window_frame.insert(0, "model", "Without TE Module")
    window_frame.insert(1, "ablation", "complete_te_module_removed")
    window_frame.insert(2, "seed", int(base_seed))
    window_frame.insert(3, "repeat_id", int(repeat_id))
    window_frame["subject_id"] = window_frame["subject"].map(
        canonical_subject_id
    )
    window_frame["prob_adhd"] = window_frame["y_prob"].astype(float)
    all_window_prediction_frames.append(window_frame)

    run_summaries.append(
        {
            "seed": int(base_seed),
            "repeat_id": int(repeat_id),
            "n_folds": int(len(fold_frame)),
            "output_dir": str(seed_output_dir),
        }
    )

fold_metrics_all = (
    pd.concat(all_fold_frames, ignore_index=True)
    .sort_values(["seed", "fold"])
    .reset_index(drop=True)
)

window_predictions_all = (
    pd.concat(all_window_prediction_frames, ignore_index=True)
    .sort_values(["seed", "fold", "subject_id", "window_id"])
    .reset_index(drop=True)
)

if fold_metrics_all["seed"].nunique() != len(BASE_SEEDS):
    raise RuntimeError("The number of completed base seeds is incorrect.")

folds_per_seed = fold_metrics_all.groupby("seed")["fold"].nunique()
if not (folds_per_seed == len(folds)).all():
    raise RuntimeError(
        "At least one base seed does not contain all five folds:\n"
        f"{folds_per_seed}"
    )

# The five folds are averaged first within each seed.
metrics_by_seed = (
    fold_metrics_all
    .groupby(["model", "ablation", "seed", "repeat_id"], as_index=False)[
        METRIC_COLUMNS
    ]
    .mean()
)

summary_rows = []
for metric in METRIC_COLUMNS:
    values = metrics_by_seed[metric].to_numpy(dtype=float)
    summary_rows.append(
        {
            "model": "Without TE Module",
            "ablation": "complete_te_module_removed",
            "metric": metric.replace("test_window_", ""),
            "mean": float(np.nanmean(values)),
            "std_sample": (
                float(np.nanstd(values, ddof=1))
                if len(values) > 1
                else 0.0
            ),
            "mean_percent": float(100.0 * np.nanmean(values)),
            "std_sample_percent": (
                float(100.0 * np.nanstd(values, ddof=1))
                if len(values) > 1
                else 0.0
            ),
            "n_seeds": int(len(values)),
        }
    )

metrics_summary = pd.DataFrame(summary_rows)
metrics_summary["Result (%)"] = metrics_summary.apply(
    lambda row: (
        f"{row['mean_percent']:.1f} "
        f"$\\pm$ {row['std_sample_percent']:.1f}"
    ),
    axis=1,
)

fold_metrics_all.to_csv(
    OUTPUT_ROOT / "WithoutTE_window_level_metrics_by_fold.csv",
    index=False,
)
metrics_by_seed.to_csv(
    OUTPUT_ROOT / "WithoutTE_window_level_metrics_by_seed.csv",
    index=False,
)
metrics_summary.to_csv(
    OUTPUT_ROOT / "WithoutTE_window_level_metrics_mean_std.csv",
    index=False,
)
window_predictions_all.to_csv(
    OUTPUT_ROOT / "WithoutTE_all_window_predictions.csv",
    index=False,
)

with open(
    OUTPUT_ROOT / "WithoutTE_repeated_protocol_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "model": "Without TE Module",
            "ablation": "complete_te_module_removed",
            "base_seeds": [int(value) for value in BASE_SEEDS],
            "n_folds": int(len(folds)),
            "n_training_runs": int(len(BASE_SEEDS) * len(folds)),
            "model_initialization_seed": 42,
            "dataloader_seed_rule": "base_seed + fold_id",
            "aggregation_rule": (
                "mean across five test folds within each seed, followed "
                "by mean and sample standard deviation across seeds"
            ),
            "run_summaries": run_summaries,
        },
        file,
        indent=2,
    )



REPETITION 1/10 | base seed = 0

FOLD 1/5
Ventanas | train=4977, validation=1574, test=1662
Sujetos  | train=76, validation=20, test=24
Epoch 001 | train_loss=0.34531 | val_loss=1.21251 | val_acc_window=0.7039 | patience=0/25
Epoch 002 | train_loss=0.13207 | val_loss=0.94557 | val_acc_window=0.7516 | patience=0/25
Epoch 004 | train_loss=0.03838 | val_loss=0.85910 | val_acc_window=0.8278 | patience=0/25
Epoch 005 | train_loss=0.03997 | val_loss=0.92530 | val_acc_window=0.8316 | patience=1/25
Epoch 010 | train_loss=0.01396 | val_loss=0.84398 | val_acc_window=0.8208 | patience=0/25
Epoch 015 | train_loss=0.00009 | val_loss=2.08769 | val_acc_window=0.8056 | patience=5/25
Epoch 020 | train_loss=0.00002 | val_loss=2.28878 | val_acc_window=0.8037 | patience=10/25
Epoch 025 | train_loss=0.00001 | val_loss=2.37657 | val_acc_window=0.8037 | patience=15/25
Epoch 030 | train_loss=0.00001 | val_loss=2.34979 | val_acc_window=0.8056 | patience=20/25
Epoch 035 | train_loss=0.00001 | val_loss=2.32594 

In [8]:
# ============================================================
# 8. PARTICIPANT-LEVEL EXPORTS FOR SUBSEQUENT PAIRED ANALYSES
# ============================================================
# These exports do not replace the window-level ablation result. They retain
# one out-of-fold participant probability per seed for later McNemar or
# bootstrap analyses if requested.

label_counts = (
    window_predictions_all
    .groupby(["seed", "subject_id"])["y_true"]
    .nunique()
)
if not (label_counts == 1).all():
    raise RuntimeError("At least one participant has inconsistent labels.")

fold_counts_subject = (
    window_predictions_all
    .groupby(["seed", "subject_id"])["fold"]
    .nunique()
)
if not (fold_counts_subject == 1).all():
    raise RuntimeError(
        "At least one participant appears in more than one test fold "
        "within the same seed."
    )

subject_predictions_by_seed = (
    window_predictions_all
    .groupby(["model", "ablation", "seed", "subject_id"], as_index=False)
    .agg(
        fold=("fold", "first"),
        y_true=("y_true", "first"),
        prob_adhd=("prob_adhd", "mean"),
        n_windows=("window_id", "size"),
    )
)
subject_predictions_by_seed["y_pred"] = (
    subject_predictions_by_seed["prob_adhd"] >= 0.5
).astype(int)
subject_predictions_by_seed["correct"] = (
    subject_predictions_by_seed["y_true"]
    == subject_predictions_by_seed["y_pred"]
).astype(int)

subjects_per_seed = subject_predictions_by_seed.groupby("seed")[
    "subject_id"
].nunique()
if subjects_per_seed.nunique() != 1:
    raise RuntimeError(
        "The number of out-of-fold participants differs across seeds:\n"
        f"{subjects_per_seed}"
    )

subject_metric_rows = []
for seed_value, group in subject_predictions_by_seed.groupby("seed"):
    values = binary_window_metrics(
        y_true=group["y_true"],
        y_pred=group["y_pred"],
        y_prob=group["prob_adhd"],
    )
    subject_metric_rows.append(
        {
            "model": "Without TE Module",
            "ablation": "complete_te_module_removed",
            "seed": int(seed_value),
            "n_subjects": int(len(group)),
            **{
                key: value
                for key, value in values.items()
                if key != "n_windows"
            },
        }
    )

subject_metrics_by_seed = pd.DataFrame(subject_metric_rows)

subject_consensus_predictions = (
    subject_predictions_by_seed
    .groupby(["model", "ablation", "subject_id"], as_index=False)
    .agg(
        y_true=("y_true", "first"),
        prob_adhd=("prob_adhd", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
subject_consensus_predictions["y_pred"] = (
    subject_consensus_predictions["prob_adhd"] >= 0.5
).astype(int)
subject_consensus_predictions["correct"] = (
    subject_consensus_predictions["y_true"]
    == subject_consensus_predictions["y_pred"]
).astype(int)

subject_predictions_by_seed.to_csv(
    OUTPUT_ROOT / "WithoutTE_subject_level_predictions_by_seed.csv",
    index=False,
)
subject_metrics_by_seed.to_csv(
    OUTPUT_ROOT / "WithoutTE_subject_level_metrics_by_seed.csv",
    index=False,
)
subject_consensus_predictions.to_csv(
    OUTPUT_ROOT / "WithoutTE_subject_level_consensus_predictions.csv",
    index=False,
)


In [9]:
# ============================================================
# 9. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 88)
print("ABLATION WITHOUT TE MODULE - REPEATED TEST RESULTS")
print("FIVE-FOLD MEAN WITHIN SEED; MEAN ± SAMPLE SD ACROSS SEEDS")
print("=" * 88)

display_labels = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced accuracy",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "precision": "Precision",
    "f1": "F1-score",
    "auc": "ROC-AUC",
    "kappa": "Kappa",
}

for metric_name, label in display_labels.items():
    row = metrics_summary.loc[
        metrics_summary["metric"] == metric_name
    ].iloc[0]
    print(
        f"{label:18s}: "
        f"{row['mean_percent']:.1f} "
        f"± {row['std_sample_percent']:.1f}%"
    )

print("\nProtocol:")
print(f"- Base seeds: {BASE_SEEDS}")
print(f"- Folds per seed: {len(folds)}")
print(f"- Total trained/evaluated models: {len(BASE_SEEDS) * len(folds)}")
print("- Model initialization seed fixed at 42, matching presented CTE-Net")
print("- DataLoader seed: base_seed + fold_id")
print("- Sample SD calculated across seed-level five-fold means (ddof=1)")
print("- Window and participant predictions retained")

print("\nFiles saved in:")
print(OUTPUT_ROOT)

display(
    metrics_summary[
        ["metric", "Result (%)", "n_seeds"]
    ]
)

display(metrics_by_seed)
display(subject_metrics_by_seed)



ABLATION WITHOUT TE MODULE - REPEATED TEST RESULTS
FIVE-FOLD MEAN WITHIN SEED; MEAN ± SAMPLE SD ACROSS SEEDS
Accuracy          : 78.7 ± 0.8%
Balanced accuracy : 78.3 ± 0.9%
Sensitivity       : 82.3 ± 2.8%
Specificity       : 74.2 ± 3.4%
Precision         : 80.8 ± 1.6%
F1-score          : 81.0 ± 1.0%
ROC-AUC           : 84.8 ± 1.5%
Kappa             : 56.6 ± 1.7%

Protocol:
- Base seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
- Folds per seed: 5
- Total trained/evaluated models: 50
- Model initialization seed fixed at 42, matching presented CTE-Net
- DataLoader seed: base_seed + fold_id
- Sample SD calculated across seed-level five-fold means (ddof=1)
- Window and participant predictions retained

Files saved in:
/kaggle/working/ablation_without_te_repeated_10seeds


,metric,Result (%),n_seeds
0,accuracy,78.7 $\pm$ 0.8,10
1,balanced_accuracy,78.3 $\pm$ 0.9,10
2,sensitivity,82.3 $\pm$ 2.8,10
3,specificity,74.2 $\pm$ 3.4,10
4,precision,80.8 $\pm$ 1.6,10
5,f1,81.0 $\pm$ 1.0,10
6,auc,84.8 $\pm$ 1.5,10
7,kappa,56.6 $\pm$ 1.7,10


,model,ablation,seed,repeat_id,test_window_accuracy,test_window_balanced_accuracy,test_window_sensitivity,test_window_specificity,test_window_precision,test_window_f1,test_window_auc,test_window_kappa
0,Without TE Module,complete_te_module_removed,0,0,0.793671,0.792974,0.808657,0.777291,0.825244,0.811651,0.857404,0.583448
1,Without TE Module,complete_te_module_removed,1,1,0.786013,0.780267,0.846745,0.713789,0.795871,0.814990,0.866610,0.562728
2,Without TE Module,complete_te_module_removed,2,2,0.786731,0.781656,0.835123,0.728189,0.801728,0.814755,0.838140,0.564211
3,Without TE Module,complete_te_module_removed,3,3,0.778100,0.771422,0.841890,0.700953,0.789154,0.808081,0.829482,0.545947
4,Without TE Module,complete_te_module_removed,4,4,0.780498,0.781225,0.791879,0.770572,0.815288,0.796950,0.832816,0.559304
5,Without TE Module,complete_te_module_removed,5,5,0.785668,0.786221,0.781161,0.791281,0.833289,0.799075,0.870446,0.568952
6,Without TE Module,complete_te_module_removed,6,6,0.783994,0.776253,0.858316,0.694190,0.786900,0.815070,0.855707,0.557142
7,Without TE Module,complete_te_module_removed,7,7,0.774467,0.770199,0.804931,0.735467,0.797962,0.796689,0.829430,0.541869
8,Without TE Module,complete_te_module_removed,8,8,0.802265,0.796408,0.855880,0.736936,0.809343,0.828077,0.858538,0.596000
9,Without TE Module,complete_te_module_removed,9,9,0.794365,0.792483,0.809752,0.775213,0.826729,0.811592,0.844082,0.584249


,model,ablation,seed,n_subjects,accuracy,balanced_accuracy,f1,sensitivity,specificity,precision,kappa,auc,tn,fp,fn,tp
0,Without TE Module,complete_te_module_removed,0,120,0.833333,0.833333,0.836066,0.850000,0.816667,0.822581,0.666667,0.871667,49,11,9,51
1,Without TE Module,complete_te_module_removed,1,120,0.808333,0.808333,0.821705,0.883333,0.733333,0.768116,0.616667,0.876389,44,16,7,53
2,Without TE Module,complete_te_module_removed,2,120,0.783333,0.783333,0.796875,0.850000,0.716667,0.750000,0.566667,0.860556,43,17,9,51
3,Without TE Module,complete_te_module_removed,3,120,0.808333,0.808333,0.824427,0.900000,0.716667,0.760563,0.616667,0.841944,43,17,6,54
4,Without TE Module,complete_te_module_removed,4,120,0.825000,0.825000,0.829268,0.850000,0.800000,0.809524,0.650000,0.860000,48,12,9,51
5,Without TE Module,complete_te_module_removed,5,120,0.816667,0.816667,0.819672,0.833333,0.800000,0.806452,0.633333,0.905278,48,12,10,50
6,Without TE Module,complete_te_module_removed,6,120,0.816667,0.816667,0.830769,0.900000,0.733333,0.771429,0.633333,0.864722,44,16,6,54
7,Without TE Module,complete_te_module_removed,7,120,0.816667,0.816667,0.825397,0.866667,0.766667,0.787879,0.633333,0.858611,46,14,8,52
8,Without TE Module,complete_te_module_removed,8,120,0.841667,0.841667,0.852713,0.916667,0.766667,0.797101,0.683333,0.888056,46,14,5,55
9,Without TE Module,complete_te_module_removed,9,120,0.858333,0.858333,0.864000,0.900000,0.816667,0.830769,0.716667,0.889167,49,11,6,54
